In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from pathlib import Path
import pandas as pd
import json
import zipfile

BASE = Path("/content/drive/MyDrive/New Jurnal Cross")

required_files = {
    "emotion2vec_manifest": BASE / "processed_intra_features_e2v_plus_base" / "environment_manifest.json",
    "emotion2vec_feature_hash": BASE / "processed_intra_features_e2v_plus_base" / "generated_feature_file_manifest_sha256.csv",

    "sequence_manifest": BASE / "processed_intra_sequence_features_plus_base" / "sequence_environment_manifest.json",
    "sequence_feature_hash": BASE / "processed_intra_sequence_features_plus_base" / "generated_sequence_feature_file_manifest_sha256.csv",

    "intra_emotion2vec": BASE / "results_intra_emotion2vec_mlp_plus_base" / "summary_mean_std.csv",
    "intra_concat": BASE / "results_intra_concat_fusion_mlp_plus_base" / "summary_mean_std.csv",
    "intra_sequence": BASE / "results_intra_sequence_cross_attention_plus_base" / "summary_mean_std.csv",

    "lodo_utterance_table": BASE / "results_lodo_utterance_fusion_plus_base" / "lodo_paper_table_test.csv",
    "lodo_utterance_all_seed": BASE / "results_lodo_utterance_fusion_plus_base" / "lodo_all_seed_results_with_handcrafted.csv",
    "lodo_utterance_manifest": BASE / "results_lodo_utterance_fusion_plus_base" / "lodo_split_manifest.csv",
    "lodo_utterance_leakage": BASE / "results_lodo_utterance_fusion_plus_base" / "lodo_leakage_prevention_checklist.csv",

    "lodo_sequence_table": BASE / "results_lodo_sequence_cross_attention_plus_base" / "lodo_sequence_xattn_paper_table_test.csv",
    "lodo_compare_all_models": BASE / "results_lodo_sequence_cross_attention_plus_base" / "compare_lodo_all_models_paper_table.csv",
    "lodo_sequence_all_seed": BASE / "results_lodo_sequence_cross_attention_plus_base" / "lodo_sequence_xattn_all_seed_results.csv",
    "lodo_sequence_manifest": BASE / "results_lodo_sequence_cross_attention_plus_base" / "lodo_sequence_split_manifest.csv",
    "lodo_sequence_leakage": BASE / "results_lodo_sequence_cross_attention_plus_base" / "lodo_sequence_leakage_prevention_checklist.csv",

    "duration_by_dataset": BASE / "processed_intra_features_e2v_plus_base" / "duration_crop_padding_stats_by_dataset.csv",
    "duration_by_dataset_emotion": BASE / "processed_intra_features_e2v_plus_base" / "duration_crop_padding_stats_by_dataset_emotion.csv",
}

check = []
for name, path in required_files.items():
    check.append({
        "name": name,
        "exists": path.exists(),
        "path": str(path)
    })

check_df = pd.DataFrame(check)
display(check_df)

missing = check_df[check_df["exists"] == False]
if len(missing) > 0:
    print("Missing files:")
    display(missing)
else:
    print("All required files exist.")

,name,exists,path
0,emotion2vec_manifest,True,/content/drive/MyDrive/New Jurnal Cross/proces...
1,emotion2vec_feature_hash,True,/content/drive/MyDrive/New Jurnal Cross/proces...
2,sequence_manifest,True,/content/drive/MyDrive/New Jurnal Cross/proces...
3,sequence_feature_hash,True,/content/drive/MyDrive/New Jurnal Cross/proces...
4,intra_emotion2vec,True,/content/drive/MyDrive/New Jurnal Cross/result...
5,intra_concat,True,/content/drive/MyDrive/New Jurnal Cross/result...
6,intra_sequence,True,/content/drive/MyDrive/New Jurnal Cross/result...
7,lodo_utterance_table,True,/content/drive/MyDrive/New Jurnal Cross/result...
8,lodo_utterance_all_seed,True,/content/drive/MyDrive/New Jurnal Cross/result...
9,lodo_utterance_manifest,True,/content/drive/MyDrive/New Jurnal Cross/result...


All required files exist.


In [6]:
manifest_path = BASE / "processed_intra_features_e2v_plus_base" / "environment_manifest.json"

with open(manifest_path, "r") as f:
    manifest = json.load(f)

print("Model ID:", manifest.get("model_id"))
print("Checkpoint path:", manifest.get("checkpoint_path"))
print("Checkpoint SHA256:", manifest.get("checkpoint_sha256"))
print("FunASR:", manifest.get("funasr_version"))
print("ModelScope:", manifest.get("modelscope_version"))
print("Torch:", manifest.get("torch_version"))
print("Created UTC:", manifest.get("created_utc"))

Model ID: iic/emotion2vec_plus_base
Checkpoint path: /root/.cache/modelscope/models/iic--emotion2vec_plus_base/snapshots/master/model.pt
Checkpoint SHA256: 60710b5aae1dbe69bdac8920028fb05882d4314fd09031922b4b61ee9e7aadbd
FunASR: 1.4.1
ModelScope: 1.39.1
Torch: 2.11.0+cu128
Created UTC: 2026-08-08T09:45:54.531915+00:00


In [7]:
lodo_table = pd.read_csv(
    BASE / "results_lodo_sequence_cross_attention_plus_base" / "compare_lodo_all_models_paper_table.csv"
)

display(lodo_table)

,Model,Held-out Test Dataset,Accuracy,UAR,Macro-F1,Weighted-F1
0,Handcrafted SVM-RBF,EMODB,27.30 ± 0.00,25.84 ± 0.00,19.74 ± 0.00,20.60 ± 0.00
1,Handcrafted SVM-RBF,RAVDESS,22.16 ± 0.00,20.31 ± 0.00,14.16 ± 0.00,15.44 ± 0.00
2,Handcrafted SVM-RBF,RESD,23.46 ± 0.00,24.60 ± 0.00,22.59 ± 0.00,22.32 ± 0.00
3,Handcrafted MLP,EMODB,28.51 ± 0.85,27.02 ± 1.13,19.80 ± 1.43,20.31 ± 1.59
4,Handcrafted MLP,RAVDESS,25.95 ± 2.55,23.78 ± 2.34,16.27 ± 3.11,17.75 ± 3.39
5,Handcrafted MLP,RESD,23.76 ± 0.05,24.61 ± 0.34,22.77 ± 0.71,22.75 ± 0.84
6,emotion2vec MLP,EMODB,83.98 ± 0.50,83.81 ± 0.52,83.52 ± 0.52,83.63 ± 0.55
7,emotion2vec MLP,RAVDESS,94.73 ± 0.39,94.99 ± 0.28,94.52 ± 0.42,94.76 ± 0.38
8,emotion2vec MLP,RESD,59.38 ± 0.27,58.43 ± 0.41,59.31 ± 0.34,59.63 ± 0.26
9,Concat Fusion MLP,EMODB,83.19 ± 0.98,83.07 ± 1.07,82.60 ± 1.19,82.72 ± 1.12


In [8]:
output_zip = BASE / "final_outputs_for_revision_check.zip"

files_to_zip = [path for path in required_files.values() if path.exists()]

with zipfile.ZipFile(output_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for path in files_to_zip:
        z.write(path, arcname=str(path.relative_to(BASE)))

print("Saved:", output_zip)
print("Number of files:", len(files_to_zip))

Saved: /content/drive/MyDrive/New Jurnal Cross/final_outputs_for_revision_check.zip
Number of files: 18


In [3]:
from pathlib import Path
import pandas as pd

BASE = Path("/content/drive/MyDrive/New Jurnal Cross")

print("BASE exists:", BASE.exists())
print("BASE:", BASE)

if BASE.exists():
    print("\nTop-level folders/files:")
    for p in sorted(BASE.iterdir()):
        print("DIR " if p.is_dir() else "FILE", p.name)

BASE exists: True
BASE: /content/drive/MyDrive/New Jurnal Cross

Top-level folders/files:
DIR  ExperimentA
DIR  ExperimentB
DIR  code
DIR  dataset
DIR  figures_plus_base
DIR  newcode
DIR  processed_intra_csv
DIR  processed_intra_features_e2v
DIR  processed_intra_features_e2v_plus_base
DIR  processed_intra_features_hc
DIR  processed_intra_features_hc_noaug
DIR  processed_intra_sequence_features
DIR  processed_intra_sequence_features_plus_base
DIR  results_intra_concat_fusion_mlp
DIR  results_intra_concat_fusion_mlp_plus_base
DIR  results_intra_cross_attention_fusion
DIR  results_intra_cross_attention_fusion_plus_base
DIR  results_intra_emotion2vec_mlp
DIR  results_intra_emotion2vec_mlp_plus_base
DIR  results_intra_handcrafted_mlp
DIR  results_intra_handcrafted_svm
DIR  results_intra_sequence_cross_attention
DIR  results_intra_sequence_cross_attention_plus_base
DIR  results_lodo_sequence_cross_attention
DIR  results_lodo_sequence_cross_attention_plus_base
DIR  results_lodo_utterance_fusi

In [4]:
from pathlib import Path
import pandas as pd

BASE = Path("/content/drive/MyDrive/New Jurnal Cross")

target_names = [
    "environment_manifest.json",
    "sequence_environment_manifest.json",
    "generated_feature_file_manifest_sha256.csv",
    "generated_sequence_feature_file_manifest_sha256.csv",
    "summary_mean_std.csv",
    "lodo_paper_table_test.csv",
    "lodo_all_seed_results_with_handcrafted.csv",
    "lodo_split_manifest.csv",
    "lodo_leakage_prevention_checklist.csv",
    "lodo_sequence_xattn_paper_table_test.csv",
    "compare_lodo_all_models_paper_table.csv",
    "lodo_sequence_xattn_all_seed_results.csv",
    "lodo_sequence_split_manifest.csv",
    "lodo_sequence_leakage_prevention_checklist.csv",
    "duration_crop_padding_stats_by_dataset.csv",
    "duration_crop_padding_stats_by_dataset_emotion.csv",
]

found = []

for name in target_names:
    for path in BASE.rglob(name):
        found.append({
            "file_name": name,
            "found_path": str(path),
            "parent_folder": path.parent.name,
            "size_bytes": path.stat().st_size
        })

found_df = pd.DataFrame(found)

if len(found_df) == 0:
    print("Tidak ada target output yang ditemukan.")
else:
    display(found_df.sort_values(["file_name", "found_path"]))

,file_name,found_path,parent_folder,size_bytes
35,compare_lodo_all_models_paper_table.csv,/content/drive/MyDrive/New Jurnal Cross/newcod...,results_lodo_sequence_cross_attention_base,1300
33,compare_lodo_all_models_paper_table.csv,/content/drive/MyDrive/New Jurnal Cross/result...,results_lodo_sequence_cross_attention,818
34,compare_lodo_all_models_paper_table.csv,/content/drive/MyDrive/New Jurnal Cross/result...,results_lodo_sequence_cross_attention_plus_base,1300
43,duration_crop_padding_stats_by_dataset.csv,/content/drive/MyDrive/New Jurnal Cross/proces...,processed_intra_features_e2v_plus_base,608
44,duration_crop_padding_stats_by_dataset_emotion...,/content/drive/MyDrive/New Jurnal Cross/proces...,processed_intra_features_e2v_plus_base,2939
1,environment_manifest.json,/content/drive/MyDrive/New Jurnal Cross/newcod...,processed_intra_features_e2v_base,587
0,environment_manifest.json,/content/drive/MyDrive/New Jurnal Cross/proces...,processed_intra_features_e2v_plus_base,2177
3,generated_feature_file_manifest_sha256.csv,/content/drive/MyDrive/New Jurnal Cross/proces...,processed_intra_features_e2v_plus_base,9705
4,generated_sequence_feature_file_manifest_sha25...,/content/drive/MyDrive/New Jurnal Cross/proces...,processed_intra_sequence_features_plus_base,11857
25,lodo_all_seed_results_with_handcrafted.csv,/content/drive/MyDrive/New Jurnal Cross/newcod...,results_lodo_utterance_fusion_base,9645


In [9]:
from pathlib import Path
import pandas as pd
import json

BASE = Path("/content/drive/MyDrive/New Jurnal Cross")

required_files = {
    "emotion2vec_manifest": BASE / "processed_intra_features_e2v_plus_base" / "environment_manifest.json",
    "emotion2vec_feature_hash": BASE / "processed_intra_features_e2v_plus_base" / "generated_feature_file_manifest_sha256.csv",

    "sequence_manifest": BASE / "processed_intra_sequence_features_plus_base" / "sequence_environment_manifest.json",
    "sequence_feature_hash": BASE / "processed_intra_sequence_features_plus_base" / "generated_sequence_feature_file_manifest_sha256.csv",

    "intra_emotion2vec": BASE / "results_intra_emotion2vec_mlp_plus_base" / "summary_mean_std.csv",
    "intra_concat": BASE / "results_intra_concat_fusion_mlp_plus_base" / "summary_mean_std.csv",
    "intra_sequence": BASE / "results_intra_sequence_cross_attention_plus_base" / "summary_mean_std.csv",

    "lodo_utterance_table": BASE / "results_lodo_utterance_fusion_plus_base" / "lodo_paper_table_test.csv",
    "lodo_utterance_all_seed": BASE / "results_lodo_utterance_fusion_plus_base" / "lodo_all_seed_results_with_handcrafted.csv",
    "lodo_utterance_manifest": BASE / "results_lodo_utterance_fusion_plus_base" / "lodo_split_manifest.csv",
    "lodo_utterance_leakage": BASE / "results_lodo_utterance_fusion_plus_base" / "lodo_leakage_prevention_checklist.csv",

    "lodo_sequence_table": BASE / "results_lodo_sequence_cross_attention_plus_base" / "lodo_sequence_xattn_paper_table_test.csv",
    "lodo_compare_all_models": BASE / "results_lodo_sequence_cross_attention_plus_base" / "compare_lodo_all_models_paper_table.csv",
    "lodo_sequence_all_seed": BASE / "results_lodo_sequence_cross_attention_plus_base" / "lodo_sequence_xattn_all_seed_results.csv",
    "lodo_sequence_manifest": BASE / "results_lodo_sequence_cross_attention_plus_base" / "lodo_sequence_split_manifest.csv",
    "lodo_sequence_leakage": BASE / "results_lodo_sequence_cross_attention_plus_base" / "lodo_sequence_leakage_prevention_checklist.csv",

    "duration_by_dataset": BASE / "processed_intra_features_e2v_plus_base" / "duration_crop_padding_stats_by_dataset.csv",
    "duration_by_dataset_emotion": BASE / "processed_intra_features_e2v_plus_base" / "duration_crop_padding_stats_by_dataset_emotion.csv",
}

check_df = pd.DataFrame([
    {"name": name, "exists": path.exists(), "path": str(path)}
    for name, path in required_files.items()
])

display(check_df)

missing = check_df[check_df["exists"] == False]
if len(missing) == 0:
    print("All required files exist.")
else:
    print("Missing files:")
    display(missing)

,name,exists,path
0,emotion2vec_manifest,True,/content/drive/MyDrive/New Jurnal Cross/proces...
1,emotion2vec_feature_hash,True,/content/drive/MyDrive/New Jurnal Cross/proces...
2,sequence_manifest,True,/content/drive/MyDrive/New Jurnal Cross/proces...
3,sequence_feature_hash,True,/content/drive/MyDrive/New Jurnal Cross/proces...
4,intra_emotion2vec,True,/content/drive/MyDrive/New Jurnal Cross/result...
5,intra_concat,True,/content/drive/MyDrive/New Jurnal Cross/result...
6,intra_sequence,True,/content/drive/MyDrive/New Jurnal Cross/result...
7,lodo_utterance_table,True,/content/drive/MyDrive/New Jurnal Cross/result...
8,lodo_utterance_all_seed,True,/content/drive/MyDrive/New Jurnal Cross/result...
9,lodo_utterance_manifest,True,/content/drive/MyDrive/New Jurnal Cross/result...


All required files exist.


In [10]:
manifest_path = BASE / "processed_intra_features_e2v_plus_base" / "environment_manifest.json"

with open(manifest_path, "r") as f:
    manifest = json.load(f)

print("Model ID:", manifest.get("model_id"))
print("Checkpoint path:", manifest.get("checkpoint_path"))
print("Checkpoint SHA256:", manifest.get("checkpoint_sha256"))
print("FunASR:", manifest.get("funasr_version"))
print("ModelScope:", manifest.get("modelscope_version"))
print("Torch:", manifest.get("torch_version"))
print("Created UTC:", manifest.get("created_utc"))

Model ID: iic/emotion2vec_plus_base
Checkpoint path: /root/.cache/modelscope/models/iic--emotion2vec_plus_base/snapshots/master/model.pt
Checkpoint SHA256: 60710b5aae1dbe69bdac8920028fb05882d4314fd09031922b4b61ee9e7aadbd
FunASR: 1.4.1
ModelScope: 1.39.1
Torch: 2.11.0+cu128
Created UTC: 2026-08-08T09:45:54.531915+00:00


In [11]:
lodo_final = pd.read_csv(
    BASE / "results_lodo_sequence_cross_attention_plus_base" / "compare_lodo_all_models_paper_table.csv"
)

display(lodo_final)

,Model,Held-out Test Dataset,Accuracy,UAR,Macro-F1,Weighted-F1
0,Handcrafted SVM-RBF,EMODB,27.30 ± 0.00,25.84 ± 0.00,19.74 ± 0.00,20.60 ± 0.00
1,Handcrafted SVM-RBF,RAVDESS,22.16 ± 0.00,20.31 ± 0.00,14.16 ± 0.00,15.44 ± 0.00
2,Handcrafted SVM-RBF,RESD,23.46 ± 0.00,24.60 ± 0.00,22.59 ± 0.00,22.32 ± 0.00
3,Handcrafted MLP,EMODB,28.51 ± 0.85,27.02 ± 1.13,19.80 ± 1.43,20.31 ± 1.59
4,Handcrafted MLP,RAVDESS,25.95 ± 2.55,23.78 ± 2.34,16.27 ± 3.11,17.75 ± 3.39
5,Handcrafted MLP,RESD,23.76 ± 0.05,24.61 ± 0.34,22.77 ± 0.71,22.75 ± 0.84
6,emotion2vec MLP,EMODB,83.98 ± 0.50,83.81 ± 0.52,83.52 ± 0.52,83.63 ± 0.55
7,emotion2vec MLP,RAVDESS,94.73 ± 0.39,94.99 ± 0.28,94.52 ± 0.42,94.76 ± 0.38
8,emotion2vec MLP,RESD,59.38 ± 0.27,58.43 ± 0.41,59.31 ± 0.34,59.63 ± 0.26
9,Concat Fusion MLP,EMODB,83.19 ± 0.98,83.07 ± 1.07,82.60 ± 1.19,82.72 ± 1.12


In [5]:
from pathlib import Path
import pandas as pd

BASE = Path("/content/drive/MyDrive/New Jurnal Cross")

folder_keywords = [
    "processed_intra_features_e2v",
    "processed_intra_sequence_features",
    "results_intra_emotion2vec",
    "results_intra_concat",
    "results_intra_sequence",
    "results_lodo_utterance",
    "results_lodo_sequence",
]

rows = []

for p in BASE.rglob("*"):
    if p.is_dir():
        for kw in folder_keywords:
            if kw in p.name:
                rows.append({
                    "folder": p.name,
                    "path": str(p)
                })

folder_df = pd.DataFrame(rows).drop_duplicates()

if len(folder_df) == 0:
    print("Tidak ada folder output yang cocok ditemukan.")
else:
    display(folder_df.sort_values("path"))

,folder,path
14,processed_intra_features_e2v_base,/content/drive/MyDrive/New Jurnal Cross/newcod...
17,processed_intra_sequence_features_base,/content/drive/MyDrive/New Jurnal Cross/newcod...
16,results_intra_concat_fusion_mlp_base,/content/drive/MyDrive/New Jurnal Cross/newcod...
15,results_intra_emotion2vec_mlp_base,/content/drive/MyDrive/New Jurnal Cross/newcod...
18,results_intra_sequence_cross_attention_base,/content/drive/MyDrive/New Jurnal Cross/newcod...
20,results_lodo_sequence_cross_attention_base,/content/drive/MyDrive/New Jurnal Cross/newcod...
19,results_lodo_utterance_fusion_base,/content/drive/MyDrive/New Jurnal Cross/newcod...
0,processed_intra_features_e2v,/content/drive/MyDrive/New Jurnal Cross/proces...
7,processed_intra_features_e2v_plus_base,/content/drive/MyDrive/New Jurnal Cross/proces...
3,processed_intra_sequence_features,/content/drive/MyDrive/New Jurnal Cross/proces...
